# Ce notebook définit les différents jobs de Lambda.

---

### **structure du projet**

TO-DO list : 

[x] Imports                                                                                                           
[x] Chargement du config.yaml
[x] Appel API → récupération du JSON (URL du ZIP)                                                                     
[x] Téléchargement du ZIP                                                                                           
[x] Extraction du CSV depuis le ZIP (en mémoire)                                                                      
[x] Aperçu des données (vérification) 
[x] Optimisation de la taille du fichier (Parquet : snappy)                                                                                
[ ] généraliser à tous les fichiers du zip.

### Etape 1 : importer les `librairies`

In [1]:
import requests #librairie permettant d'effectuer des requêtes API
import yaml #lire le fichier config.yaml qui contient l'URL_API
import io #créer un fichier virtuel
import zipfile #interpréter un ZIP
import pandas as pd
import pyarrow #pour la conversion en Parquet

### Etape 2 : Importer l'URL de `config.yaml` et écrire la requête paramétrée

1. *Ouvrir et parser le config.yaml → obtenir un dictionnaire Python*
2. *Extraire config["Lambda"]["URL_API"] → stocker dans une variable url*                                                       
3. *Passer cette variable à requests.get() avec le dictionnaire params pour le filtre année*

In [2]:
file = open("../config/config.yaml") #chemin relatif
data = yaml.safe_load(file)
file.close()

Le format renvoyé est JSON car le `YAML` est un **superset** de `JSON`

In [3]:
data

{'S3': {'bucket': 'p-idfm-pipeline',
  'chemins_dossiers': {'bronze': 'bronze/',
   'silver': 'silver/',
   'gold': 'gold/',
   'profiling': 'profiling/',
   'quality_reports': 'quality_reports/',
   'pipeline_metadata': 'pipeline_metadata/',
   'athena_results': 'athena_results/'},
  'region': 'eu-west-3'},
 'Lambda': {'fonction': 'p-idfm-pipeline-fetch-api',
  'timeout': 300,
  'memoire_allouee': 256,
  'URL_API': 'https://data.iledefrance-mobilites.fr/api/explore/v2.0/catalog/datasets/histo-validations-reseau-ferre/exports/json'},
 'Glue': {'nom_du_job': 'glue-transf-bronzetosilver',
  'timeout': 10,
  'type_worker': 'G.1X',
  'nb_workers': 2,
  'version': 5.0},
 'Athena': {'nom': 'athena-idfm', 'chemin_output': 'athena_results/'},
 'Pipeline': {'environnement': 'dev',
  'periode': '2015-2024',
  'filtre_ZdAtype': ['metroStation', 'railStation', 'liftStation']}}

On peut extraire l'url statique qu'on a extrait de IDFM. On voit que la console est la v.2.0. On va utiliser params pour filtrer sur l'année.

In [4]:
url = data['Lambda']['URL_API']

On commence la requête avec la librairie `requests`

In [5]:
r = requests.get(url)

Je check le `statut` de la requête. 200 = fonctionne

In [6]:
r.raise_for_status() # le statut de la requête renvoie 200, c'est ok

Je passe le résultat de la requête sous format `JSON`

In [7]:
api_raw_json = r.json()

Un premier test me permet de comprendre qu'il s'agit d'une **liste de dictionnaires**

In [8]:
api_raw_json[0]['reseau_ferre']['url']

'https://data.iledefrance-mobilites.fr/api/explore/v2.0/catalog/datasets/histo-validations-reseau-ferre/files/2ac7b336c2e7be2586f20bd8f298ab29'

Je crée une boucle simple pour sortir l'url de chaque année, j'utilise **`for`**

In [9]:
for i in api_raw_json :
    if i['reseau_ferre']:
        print(f"année {i['annee']} : {i['reseau_ferre']['url']}")

année 2017 : https://data.iledefrance-mobilites.fr/api/explore/v2.0/catalog/datasets/histo-validations-reseau-ferre/files/2ac7b336c2e7be2586f20bd8f298ab29
année 2019 : https://data.iledefrance-mobilites.fr/api/explore/v2.0/catalog/datasets/histo-validations-reseau-ferre/files/6d7e7b859e6acac7bebad18bdb37bfc3
année 2020 : https://data.iledefrance-mobilites.fr/api/explore/v2.0/catalog/datasets/histo-validations-reseau-ferre/files/e6bcf4c994951fc086e31db6819a3448
année 2021 : https://data.iledefrance-mobilites.fr/api/explore/v2.0/catalog/datasets/histo-validations-reseau-ferre/files/e35b9ec0a183a8f2c7a8537dd43b124c
année 2016 : https://data.iledefrance-mobilites.fr/api/explore/v2.0/catalog/datasets/histo-validations-reseau-ferre/files/03dbbf888b1bb2386a2e5452861c5028
année 2018 : https://data.iledefrance-mobilites.fr/api/explore/v2.0/catalog/datasets/histo-validations-reseau-ferre/files/e1ef1b42c0e0ff7ea62ac76937ff0a60
année 2015 : https://data.iledefrance-mobilites.fr/api/explore/v2.0/ca

On commence par extraire les données de l'année `2017`en test. On utilise **`params`**

Par défaut les données **zip** sont téléchargées sur la mémoire vive `RAM`, pas écrites sur le disque.
Les bytes du ZIP sont téléchargés et accessibles grâce à `r.content`

In [10]:
url_2017 = api_raw_json[0]['reseau_ferre']['url']
request_2017 = requests.get(url_2017)
# request_2017.content permet d'afficher les bytes

In [11]:
request_2017.raise_for_status()

On a les données brutes. Pour pouvoir "**envelopper les données dans un fichier lisible**" il faut passer par deux étapes : 
- `io.BytesIO` — transforme les bytes en un objet "fichier virtuel" en RAM
- `zipfile.ZipFile` — ouvre ce fichier et intérprète le ZIP.

In [12]:
zip_buffer = io.BytesIO(request_2017.content) #buffer en mémoire qui contient le ZIP

In [13]:
zip_file = zipfile.ZipFile(zip_buffer) #objet de ZIP ouvert <-> open("fichier.txt")

In [14]:
fichiers = zip_file.namelist() #permet d'accéder aux différents fichiers à l'intérieur du ZIP, sous forme de liste

In [15]:
fichiers_nb_fer = [i for i in fichiers if "NB_FER.txt" in i] #filtre les fichiers NB_FER uniquement (list comprehension)

In [16]:
f = zip_file.open(fichiers_nb_fer[0]) #on regarde ce qu'il y a dans notre premier fichiers, les deux premières lignes
lignes = f.readlines()
print(lignes[0])
print(lignes[1])
f.close()

b'JOUR\tCODE_STIF_TRNS\tCODE_STIF_RES\tCODE_STIF_ARRET\tLIBELLE_ARRET\tID_REFA_LDA\tCATEGORIE_TITRE\tNB_VALD\r\n'
b'01/07/2017\t100\t110\t1\tPORTE MAILLOT\t71379\t?\t155\r\n'


In [17]:
f = zip_file.open(fichiers_nb_fer[0]) #on stocke le premier fichier dans un DataFrame nommé df
df = pd.read_csv(f,
                delimiter='\t',
                encoding='UTF-8')
f.close()  

In [18]:
df.head() #on affiche les premières lignes du df

,JOUR,CODE_STIF_TRNS,CODE_STIF_RES,CODE_STIF_ARRET,LIBELLE_ARRET,ID_REFA_LDA,CATEGORIE_TITRE,NB_VALD
0,01/07/2017,100,110,1,PORTE MAILLOT,71379,?,155
1,01/07/2017,100,110,1,PORTE MAILLOT,71379,AMETHYSTE,204
2,01/07/2017,100,110,1,PORTE MAILLOT,71379,FGT,330
3,01/07/2017,100,110,1,PORTE MAILLOT,71379,IMAGINE R,1551
4,01/07/2017,100,110,1,PORTE MAILLOT,71379,NAVIGO,5958


**Exemple** : On contrôle le dataset

In [19]:
df.shape

(825698, 8)

In [20]:
df.dtypes

JOUR               object
CODE_STIF_TRNS      int64
CODE_STIF_RES      object
CODE_STIF_ARRET    object
LIBELLE_ARRET      object
ID_REFA_LDA         int64
CATEGORIE_TITRE    object
NB_VALD             int64
dtype: object

In [21]:
df.isna().sum()

JOUR               0
CODE_STIF_TRNS     0
CODE_STIF_RES      0
CODE_STIF_ARRET    0
LIBELLE_ARRET      0
ID_REFA_LDA        0
CATEGORIE_TITRE    0
NB_VALD            0
dtype: int64

`Data Structuring` :
- valider les types [x]
- optimiser la taille des fichiers (en Parquet) [x]
- organiser le stockage <- je le fais dans S3 directement [x]



In [22]:
parquet_buffer = io.BytesIO()

In [25]:
df.to_parquet(parquet_buffer, compression= 'snappy', index= False) #on convertit en Parquet pour optimiser le stockage en prod.

In [ ]:
print(parquet_buffer.tell()) #retourne le nombre de Bytes (le fichier pèse environ 5.8Mo)

5810924


In [29]:
f = zip_file.open(fichiers_nb_fer[0])
contenu_brut = f.read()
f.close()

print(len(contenu_brut))
print(len(request_2017.content))

45977574
14155069
